In [ ]:
from google.colab import drive
drive.mount('/gdrive')

In [ ]:
data_folder = "/gdrive/MyDrive/data/"

In [ ]:
!ls /gdrive/MyDrive/data

# Import Data

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv(data_folder + 'station.csv')

In [ ]:
len(df)

In [ ]:
# df = df.iloc[:-1]

In [ ]:
# len(df)

In [ ]:
df.describe()

In [ ]:
df.head()

In [ ]:
df.tail()

# Créer la série temporelle

In [ ]:
temps = df.loc[:, "JAN" : "DEC"]
temps

In [ ]:
series = []

for i , row in temps.iterrows():
  series += list(row)

In [ ]:
print(series)

In [ ]:
53*12

In [ ]:
len(series)

In [ ]:
series = series[:-4]

In [ ]:
len(series)

In [ ]:
import numpy as np

In [ ]:
series = np.array(series)

In [ ]:
series

In [ ]:
series.shape

# Data Visualization

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
time = np.arange(len(series))

In [ ]:
time

In [ ]:
plt.plot(time, series)
plt.xlabel('Temps')
plt.ylabel('Temperature')
plt.grid()
plt.show()

In [ ]:
np.argmax(series)

In [ ]:
series[197]

In [ ]:
extraits = series[196:199]
extraits

In [ ]:
extraits = np.delete(extraits, 1)
extraits

In [ ]:
np.mean(extraits)

In [ ]:
series.max()

In [ ]:
series[197] = np.mean(extraits)

In [ ]:
series[196:199]

In [ ]:
series.max()

In [ ]:
plt.plot(time, series)
plt.xlabel('Temps')
plt.ylabel('Temperature')
plt.grid()
plt.show()

# Baseline (Modèle le + simple possible)

In [ ]:
len(series)*.80

In [ ]:
# train_test_split - 532pour être env. 80% mais avoir pile 100 vals pour test

time_train = time[:532]
x_train = series[:532]

time_test = time[532:]
x_test = series[532:]

In [ ]:
years = df['YEAR'].tolist()
print (len(years), years)

In [ ]:
532/12

In [ ]:
years[44] # 1973-2016 train, 2016-2025 test

## Approche naïve

In [ ]:
pred_naive = series[531:-1]
len(pred_naive)

In [ ]:
pred_naive.shape

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
mean_absolute_error(x_test, pred_naive)

In [ ]:
mean_squared_error(x_test, pred_naive)

In [ ]:
plt.plot(time_test, x_test)
plt.plot(time_test, pred_naive)
plt.xlabel('Temps')
plt.ylabel('Temperature')
plt.grid(True)
plt.show()

# Creer un windowed dataset

In [ ]:
import tensorflow as tf

In [ ]:
dataset = tf.data.Dataset.range(10)

In [ ]:
dataset

In [ ]:
dataset = tf.data.Dataset.range(10)
dataset = dataset.window(5, shift=1, drop_remainder=True)
dataset = dataset.flat_map(lambda window: window.batch(5))

In [ ]:
for window in dataset:
  print(window.numpy())

In [ ]:
dataset = tf.data.Dataset.range(10)
dataset = dataset.window(5, shift=1, drop_remainder=True)
dataset = dataset.flat_map(lambda window: window.batch(5))
dataset = dataset.map(lambda window: (window[:-1], window[-1]))

In [ ]:
for x, y in dataset:
  print(x.numpy(), y.numpy())

In [ ]:
dataset = tf.data.Dataset.range(10)
dataset = dataset.window(5, shift=1, drop_remainder=True)
dataset = dataset.flat_map(lambda window: window.batch(5))
dataset = dataset.map(lambda window: (window[:-1], window[-1]))
dataset = dataset.shuffle(buffer_size=10)

dataset = dataset.batch(2).prefetch(1)


In [ ]:
for x, y in dataset:
  print(x.numpy(), y.numpy())

In [ ]:
dataset = tf.data.Dataset.from_tensor_slices(series)
dataset = dataset.window(5, shift=1, drop_remainder=True)
dataset = dataset.flat_map(lambda window: window.batch(5))
dataset = dataset.map(lambda window: (window[:-1], window[-1]))
dataset = dataset.shuffle(buffer_size=len(series))

dataset = dataset.batch(32).prefetch(1)

In [ ]:
def windowed_dataset(series, window_size, batch_size):
  dataset = tf.data.Dataset.from_tensor_slices(series)
  dataset = dataset.window(window_size+1, shift=1, drop_remainder=True)
  dataset = dataset.flat_map(lambda window: window.batch(window_size+1))
  dataset = dataset.map(lambda window: (window[:-1], window[-1]))
  # dataset = dataset.shuffle(buffer_size=len(series))

  dataset = dataset.batch(batch_size).prefetch(1)

  return dataset

In [ ]:
window_size = 24
batch_size = 32
dataset = windowed_dataset(x_train, window_size, batch_size)
test_dataset = windowed_dataset(x_test, window_size, batch_size)

# Réseaux de Neurones

In [ ]:
for feature, label in test_dataset.take(1):
  print(feature.shape)
  print(label.shape)

In [ ]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Dense(10, input_shape=[24], activation='relu'),
    tf.keras.layers.Dense(10,  activation='relu'),
    tf.keras.layers.Dense(1)
])

In [ ]:
model.summary()

In [ ]:
model.compile(loss='mse', optimizer=tf.keras.optimizers.SGD(learning_rate=1e-6, momentum=0.9))


In [ ]:
model.fit(dataset, epochs=100, validation_data=test_dataset)

# Prediction avec reseaux de neurones

In [ ]:
x_test

In [ ]:
series[505-24:505].shape

In [ ]:
r = model.predict(series[505-24:505].reshape(1, -1))

In [ ]:
r.item()

In [ ]:
forecast = []

for time in range(505-24, 605-24):

  series_to_predict = series[time: time+24].reshape(1, -1)
  r = model.predict(series_to_predict)
  forecast.append(r.item())


In [ ]:
print(forecast)

In [ ]:
forecast = np.array(forecast)

In [ ]:
print('mae : ', mean_absolute_error(x_test, forecast))
print('mse : ', mean_squared_error(x_test, forecast))
plt.plot(time_test, x_test)
plt.plot(time_test, forecast)
plt.xlabel('Temps')
plt.ylabel('Temperature')
plt.grid(True)
plt.show()

# Réseaux convolutionnels CNN

In [ ]:
series.shape

In [ ]:
series.reshape(-1, 1).shape

In [ ]:
def windowed_dataset(series, window_size, batch_size):
  series = series.reshape(-1, 1)
  dataset = tf.data.Dataset.from_tensor_slices(series)
  dataset = dataset.window(window_size+1, shift=1, drop_remainder=True)
  dataset = dataset.flat_map(lambda window: window.batch(window_size+1))
  dataset = dataset.map(lambda window: (window[:-1], window[-1]))
  # dataset = dataset.shuffle(buffer_size=len(series))

  dataset = dataset.batch(batch_size).prefetch(1)

  return dataset

In [ ]:
dataset = windowed_dataset(x_train, window_size, batch_size)
test_dataset = windowed_dataset(x_test, window_size, batch_size)

In [ ]:
model = tf.keras.models.Sequential([

    tf.keras.layers.Conv1D(filters=128, kernel_size=3, padding='causal', input_shape=[None, 1],
                           activation='relu'),
    tf.keras.layers.Dense(28, activation='relu'),
    tf.keras.layers.Dense(10, activation='relu'),
    tf.keras.layers.Dense(1)
])

In [ ]:
model.summary()

In [ ]:
model.compile(loss='mse', optimizer=tf.keras.optimizers.SGD(learning_rate=1e-6, momentum=0.9))
model.fit(dataset, epochs=100, validation_data=test_dataset)

In [ ]:
def predict_dataset(model, series, window_size):

  dataset = tf.data.Dataset.from_tensor_slices(series)
  dataset = dataset.window(window_size, shift=1, drop_remainder=True)
  dataset = dataset.flat_map(lambda window: window.batch(window_size))

  # dataset = dataset.shuffle(buffer_size=len(series))

  dataset = dataset.batch(32).prefetch(1)
  forecast = model.predict(dataset)

  return forecast

In [ ]:
x_test

In [ ]:
series[505-24:-1].shape

In [ ]:
cnn_forecast = predict_dataset(model, series[505-24:-1], window_size)

In [ ]:
cnn_forecast.shape

In [ ]:
results = cnn_forecast[:, -1, 0]

In [ ]:
results

In [ ]:
print('mae : ', mean_absolute_error(x_test, results))
print('mse : ', mean_squared_error(x_test, results))
plt.plot(time_test, x_test)
plt.plot(time_test, results)
plt.xlabel('Temps')
plt.ylabel('Temperature')
plt.grid(True)
plt.show()

# Reseaux de Neurones Récurrents RNN

In [ ]:
model = tf.keras.models.Sequential([

    tf.keras.layers.SimpleRNN(100, input_shape=[None, 1], return_sequences=True),
    tf.keras.layers.SimpleRNN(100),
    tf.keras.layers.Dense(1)
])

In [ ]:
huber = tf.keras.losses.Huber()
model.compile(loss=huber, optimizer=tf.keras.optimizers.SGD(learning_rate=1e-6, momentum=0.9))
model.fit(dataset, epochs=100, validation_data=test_dataset)

In [ ]:
rnn_forecast = predict_dataset(model, series[505-24:-1], window_size)

In [ ]:
rnn_forecast.shape

In [ ]:
rnn_forecast = rnn_forecast[:, 0]

In [ ]:
rnn_forecast.shape

In [ ]:
x_test.shape

In [ ]:
print('mae : ', mean_absolute_error(x_test, rnn_forecast))
print('mse : ', mean_squared_error(x_test, rnn_forecast))
plt.plot(time_test, x_test)
plt.plot(time_test, rnn_forecast)
plt.xlabel('Temps')
plt.ylabel('Temperature')
plt.grid(True)
plt.show()

# RNN Normalized

In [ ]:
x_train_scaled = (x_train - x_train.mean()) / x_train.std()
x_test_scaled = (x_test - x_train.mean()) / x_train.std()

In [ ]:
dataset = windowed_dataset(x_train_scaled, window_size, batch_size)
test_dataset = windowed_dataset(x_test_scaled, window_size, batch_size)

In [ ]:
model = tf.keras.models.Sequential([

    tf.keras.layers.SimpleRNN(100, input_shape=[None, 1], return_sequences=True),
    tf.keras.layers.SimpleRNN(100),
    tf.keras.layers.Dense(1)
])

In [ ]:
huber = tf.keras.losses.Huber()
model.compile(loss=huber, optimizer=tf.keras.optimizers.SGD(learning_rate=1e-6, momentum=0.9))
model.fit(dataset, epochs=100, validation_data=test_dataset)

In [ ]:
series_scaled = (series - series.mean()) / series.std()

In [ ]:
rnn_forecast = predict_dataset(model, series_scaled[505-24:-1], window_size)[:, 0]

In [ ]:
print('mae : ', mean_absolute_error(x_test_scaled, rnn_forecast))
print('mse : ', mean_squared_error(x_test_scaled, rnn_forecast))
plt.plot(time_test, x_test_scaled)
plt.plot(time_test, rnn_forecast)
plt.xlabel('Temps')
plt.ylabel('Temperature')
plt.grid(True)
plt.show()

# GRU

In [ ]:
model = tf.keras.models.Sequential([

    tf.keras.layers.GRU(100, input_shape=[None, 1], return_sequences=True, dropout=0.1),
    tf.keras.layers.GRU(100),
    tf.keras.layers.Dense(1)
])

In [ ]:
huber = tf.keras.losses.Huber()
model.compile(loss=huber, optimizer=tf.keras.optimizers.SGD(learning_rate=1e-6, momentum=0.9))
model.fit(dataset, epochs=100, validation_data=test_dataset)

In [ ]:
gru_forecast = predict_dataset(model, series_scaled[505-24:-1], window_size)[:, 0]

In [ ]:
print('mae : ', mean_absolute_error(x_test_scaled, gru_forecast))
print('mse : ', mean_squared_error(x_test_scaled, gru_forecast))
plt.plot(time_test, x_test_scaled)
plt.plot(time_test, gru_forecast)
plt.xlabel('Temps')
plt.ylabel('Temperature')
plt.grid(True)
plt.show()

# LSTM

In [ ]:
model = tf.keras.models.Sequential([

    tf.keras.layers.LSTM(24, input_shape=[None, 1], return_sequences=True),
    tf.keras.layers.LSTM(24),
    tf.keras.layers.Dense(1)
])

In [ ]:
huber = tf.keras.losses.Huber()
model.compile(loss=huber, optimizer=tf.keras.optimizers.SGD(learning_rate=1e-6, momentum=0.9))
model.fit(dataset, epochs=100, validation_data=test_dataset)

In [ ]:
lstm_forecast = predict_dataset(model, series_scaled[505-24:-1], window_size)[:, 0]

In [ ]:
print('mae : ', mean_absolute_error(x_test_scaled, lstm_forecast))
print('mse : ', mean_squared_error(x_test_scaled, lstm_forecast))
plt.plot(time_test, x_test_scaled)
plt.plot(time_test, lstm_forecast)
plt.xlabel('Temps')
plt.ylabel('Temperature')
plt.grid(True)
plt.show()